# Figure 4: DFT and Experimental I-Z Analysis
Comparison of DFT calculations and experimental I-Z measurements

In [1]:
import os
import numpy as np
import xarray as xr
import holoviews as hv
import hvplot.pandas
import pandas as pd
import rhkpy
import requests
from scipy.ndimage import gaussian_filter1d

hv.extension('bokeh')

In [2]:
import sys
print(f'Python version: {sys.version}')
print(f'numpy version: {np.__version__}')
print(f'xarray version: {xr.__version__}')
print(f'holoviews version: {hv.__version__}')
print(f'rhkpy version: {rhkpy.__version__}')
print(f'pandas version: {pd.__version__}')

Python version: 3.13.5 | packaged by Anaconda, Inc. | (main, Jun 12 2025, 11:23:37) [Clang 14.0.6 ]
numpy version: 2.3.1
xarray version: 2025.6.1
holoviews version: 1.21.0
rhkpy version: 1.3.9
pandas version: 2.3.0


In [3]:
# Download source files from Zenodo if missing
ZENODO_FILES = {
    'ldos_gr_2L.dat': 'https://zenodo.org/records/17469441/files/ldos_z_graphene_2L_+500meV_PREC.dat?download=1',
    'ldos_alkane.dat': 'https://zenodo.org/records/17469441/files/ldos_z_alkan_N_2_k_2_6_vancso6.dat?download=1',
    'ldos_gr_scaled.dat': 'https://zenodo.org/records/17469441/files/ldos_z_graphene_N_2_k_2_6_scaled_vancso6.dat?download=1',
    'iz_alkane.sm4': 'https://zenodo.org/records/17469441/files/Stripes-9K-HOPG-SPI2-3_2021_09_09_05_50_41_043.sm4?download=1',
    'iz_uhv.sm4': 'https://zenodo.org/records/17469441/files/I-Z_hopg_9K_2021_04_13_14_13_48_095.sm4?download=1',
    'iz_alkin.sm4': 'https://zenodo.org/records/17469441/files/MK_ABC_FLG_25_9K_2023_10_19_11_29_53_727.sm4?download=1',
}

for local, url in ZENODO_FILES.items():
    if url is None or os.path.exists(local):
        continue
    print(f'Downloading {local}...')
    response = requests.get(url, timeout=60)
    response.raise_for_status()
    with open(local, 'wb') as fh:
        fh.write(response.content)
    print(f'✓ {local} downloaded')

In [4]:
# Helper functions for data processing
def der_dft(x, y, idl):
    """Calculate derivative for DFT data"""
    x = x[idl:]
    y = y[idl:]
    zmin = np.min(x)
    z = x - zmin
    I = y
    Ilog = np.log(I)
    dI = np.gradient(Ilog, z) / (-2)
    return (z, I, Ilog, dI)

def der_single(z, I2):
    """Calculate derivative for experimental data"""
    Ilog = np.log(I2)
    dI = np.gradient(Ilog, z) / (-2)
    return (z, I2, Ilog, dI)

def deriv_err(x, yerr):
    """Calculate error propagation for derivatives"""
    err = np.ones(len(x))
    for i in range(len(x)-1):
        dx = x[i+1] - x[i]
        err[i+1] = np.sqrt((yerr[i+1]/dx)**2 + (yerr[i]/dx)**2)
    return err

In [5]:
# Load and process DFT data
# Bilayer graphene
dft_Igr1 = np.loadtxt('ldos_gr_2L.dat')
dft_zgr1 = np.linspace(0, 40, len(dft_Igr1))
dft_gr1 = der_dft(dft_zgr1, dft_Igr1, 70)
gr_df = pd.DataFrame({'z': dft_gr1[0], 'I': dft_gr1[1], 'dI': dft_gr1[3]})

# Bilayer with alkane
dft_Ia1 = np.loadtxt('ldos_alkane.dat')
dft_za1 = np.linspace(0, 40, len(dft_Ia1))
dft_a1 = der_dft(dft_za1, dft_Ia1, 344)
a_df = pd.DataFrame({'z': dft_a1[0], 'I': dft_a1[1], 'dI': dft_a1[3]})

# Bilayer without alkane (scaled)
dft_Ina1 = np.loadtxt('ldos_gr_scaled.dat')
dft_zna1 = np.linspace(0, 40, len(dft_Ina1))
dft_Ina1_filtered = gaussian_filter1d(dft_Ina1, sigma=2.4)
dft_na1 = der_dft(dft_zna1, dft_Ina1_filtered, 314)
na_df = pd.DataFrame({'z': dft_na1[0], 'I': dft_na1[1], 'dI': dft_na1[3]})

In [6]:
# DFT plot 1: Integrated DOS
uhv_color = (90/255, 165/255, 225/255)
line_width = 1.5

gr1 = na_df.hvplot(x='z', y='I', label='Clean', color=uhv_color, line_width=line_width)
alk1 = a_df.hvplot(x='z', y='I', label='Contaminated', line_width=line_width)

layout1 = (gr1 * alk1).opts(
    title='DFT: Integrated DOS',
    xlabel='Distance (Å)',
    ylabel='Integrated DOS (a.u)',
    xlim=(0, 2),
    width=520,
    height=470,
    show_grid=True,
    legend_position='right'
)
layout1

:Overlay
   .Curve.Clean        :Curve   [z]   (I)
   .Curve.Contaminated :Curve   [z]   (I)

In [7]:
# hv.save(layout1, 'fig4_dft_decay.html')

In [8]:
# DFT plot 2: DOS decay
gr2 = na_df.hvplot(x='z', y='dI', label='Clean', color=uhv_color, line_width=line_width)
alk2 = a_df.hvplot(x='z', y='dI', label='Contaminated', line_width=line_width)

layout2 = (gr2 * alk2).opts(
    title='DFT: DOS Decay',
    xlabel='Distance (Å)',
    ylabel='DOS decay (1/Å)',
    xlim=(0, 2),
    ylim=(0.9, 2.4),
    width=520,
    height=470,
    show_grid=True,
    legend_position='right'
)
layout2

:Overlay
   .Curve.Clean        :Curve   [z]   (dI)
   .Curve.Contaminated :Curve   [z]   (dI)

In [9]:
# hv.save(layout2, 'fig4_dft_kappa.html')

In [10]:
# DFT schematic plots (log scale)
idx1 = 268
zgr2 = dft_zna1[idx1:idx1 + 90]
zgr2 = zgr2 - min(zgr2)
Igr2 = dft_Ina1[idx1:idx1 + 90]
Igr2[0:10] = np.nan
dec_gr = pd.DataFrame({'z': zgr2, 'I': Igr2})

layout5 = dec_gr.hvplot(x='z', y='I', label='Clean', color=uhv_color, line_width=line_width).opts(
    title='DFT: Clean Graphene Decay',
    xlabel='Distance (Å)',
    ylabel='integrated DOS (a.u.)',
    logy=True,
    xlim=(0, 9),
    ylim=(1e-3, 1e6),
    width=340,
    height=345,
    show_grid=True
)
layout5

:Curve   [z]   (I)

In [11]:
# hv.save(layout5, 'fig4_sch_gr.html')

In [12]:
# Alkane schematic
za2 = dft_za1[idx1:idx1 + 140]
za2 = za2 - min(za2)
Ia2 = dft_Ia1[idx1:idx1 + 140]
Ia2[0:10] = np.nan
dec_a = pd.DataFrame({'z': za2, 'I': Ia2})

red_color = (232/255, 92/255, 63/255)
layout6 = dec_a.hvplot(x='z', y='I', color=red_color, line_width=line_width).opts(
    title='DFT: Graphite + Alkane Decay',
    xlabel='Distance (Å)',
    ylabel='integrated DOS (a.u.)',
    logy=True,
    xlim=(0, 9),
    ylim=(1e-3, 1e6),
    width=340,
    height=340,
    show_grid=True
)
layout6

:Curve   [z]   (I)

In [13]:
# hv.save(layout6, 'fig4_sch_a.html')

In [14]:
(layout5.relabel("graphite") * layout6.relabel("graphite and alkane")).opts(
    title='DFT: Decay comparison',
    legend_position='top',
    show_legend=True,
    height=400,
    width=400,
)

:Overlay
   .Curve.Graphite            :Curve   [z]   (I)
   .Curve.Graphite_and_alkane :Curve   [z]   (I)

In [15]:
# Load experimental data - Alkane contaminated (80 pA)
iz_map_out = rhkpy.rhkdata('iz_alkane.sm4')
z_single = iz_map_out.spectra.variables['z'].to_numpy() * 10
I_single = abs(iz_map_out.spectra.variables['current']).mean(dim=['repetitions', 'zscandir']).to_numpy()[:, 8, 18]
I_alk_f = abs(iz_map_out.spectra.variables['current']).to_numpy()[:, 8, 18]

# Calculate error
stds_alk = []
for i in range(10):
    for j in range(2):
        stds_alk.append(np.std(I_alk_f[27:, i, j]))

iz_alk = der_single(z_single, I_single)
std_alk = np.mean(stds_alk) / np.sqrt(20)
alk_err_lin = np.ones(len(iz_alk[1])) * std_alk * 2
alk_err_log = alk_err_lin * (1 / gaussian_filter1d(iz_alk[1], 0.1))
alk_err_der = deriv_err(iz_alk[0], gaussian_filter1d(alk_err_log, 0.1))
alk_err_der[0] = alk_err_der[1] - (alk_err_der[2] - alk_err_der[1])

exp_alk_df = pd.DataFrame({
    'z': iz_alk[0], 'I': iz_alk[1], 'dI': iz_alk[3],
    'err1': alk_err_lin, 'err2': alk_err_der
})

In [16]:
# Load experimental data - UHV cleaved
uhv_data = rhkpy.rhkdata('iz_uhv.sm4').spectra
z_single_uhv = abs(uhv_data.variables['z'].to_numpy()) * 10
I_single_uhv = abs(uhv_data.variables['current']).mean(dim=['repetitions', 'zscandir']).to_numpy()
I_single_uhv = I_single_uhv - 4.4105
I_uhv_f = abs(uhv_data.variables['current']).to_numpy()

# Calculate error
stds_uhv = []
for i in range(10):
    for j in range(2):
        stds_uhv.append(np.std(I_uhv_f[17:, i, j]))

iz_uhv = der_single(z_single_uhv, I_single_uhv)
std_uhv = np.mean(stds_uhv) / np.sqrt(20)
uhv_err_lin = np.ones(len(iz_uhv[1])) * std_uhv * 2
uhv_err_log = uhv_err_lin * (1 / gaussian_filter1d(iz_uhv[1], 0.1))
uhv_err_der = deriv_err(iz_uhv[0], gaussian_filter1d(uhv_err_log, 0.1))
uhv_err_der[0] = uhv_err_der[1] - (uhv_err_der[2] - uhv_err_der[1])

exp_uhv_df = pd.DataFrame({
    'z': iz_uhv[0], 'I': iz_uhv[1], 'dI': iz_uhv[3],
    'err1': uhv_err_lin, 'err2': uhv_err_der
})

In [17]:
# Load experimental data - Contaminated (150 pA)
iz_map_in2 = rhkpy.rhkdata('iz_alkin.sm4').spectra
z_single_alkin = abs(iz_map_in2.variables['z'].to_numpy()) * 10
I_single_alkin = abs(iz_map_in2.variables['current']).mean(dim=['repetitions', 'zscandir']).to_numpy()[:, 5, 32]
I_alkin_f = abs(iz_map_in2.variables['current']).to_numpy()[:, 5, 32]

# Calculate error
stds_alkin = []
for i in range(5):
    for j in range(2):
        stds_alkin.append(np.std(I_alkin_f[56:, i, j]))

iz_alkin = der_single(z_single_alkin, I_single_alkin)
std_alkin = np.mean(stds_alkin) / np.sqrt(10)
alkin_err_lin = np.ones(len(iz_alkin[1])) * std_alkin * 2
alkin_err_log = alkin_err_lin * (1 / gaussian_filter1d(iz_alkin[1], 0.1))
alkin_err_der = deriv_err(iz_alkin[0], gaussian_filter1d(alkin_err_log, 0.1))
alkin_err_der[0] = alkin_err_der[1] - (alkin_err_der[2] - alkin_err_der[1])

exp_alkin_df = pd.DataFrame({
    'z': iz_alkin[0], 'I': iz_alkin[1], 'dI': iz_alkin[3],
    'err1': alkin_err_lin, 'err2': alkin_err_der
})

In [18]:
# Experimental plot 1: Current vs distance with error bars
gold_color = (221/255, 176/255, 80/255)

exp_uhv1 = exp_uhv_df.hvplot(x='z', y='I', label='Clean', color=uhv_color, line_width=line_width)
exp_alk1 = exp_alk_df.hvplot(x='z', y='I', label='Cont., 80 pA', line_width=line_width)
exp_alkin1 = exp_alkin_df.hvplot(x='z', y='I', label='Cont., 150 pA', color=gold_color, line_width=line_width)

# Create error bars
errors_uhv1 = [(iz_uhv[0][i], iz_uhv[1][i], uhv_err_lin[i]) for i in range(len(uhv_err_lin))]
errors_alk1 = [(iz_alk[0][i], iz_alk[1][i], alk_err_lin[i]) for i in range(len(alk_err_lin))]
errors_alkin1 = [(iz_alkin[0][i], iz_alkin[1][i], alkin_err_lin[i]) for i in range(len(alkin_err_lin))]

err_uhv1 = hv.ErrorBars(errors_uhv1).opts(color=uhv_color, line_width=0.35)
err_alk1 = hv.ErrorBars(errors_alk1).opts(color=red_color, line_width=0.35)
err_alkin1 = hv.ErrorBars(errors_alkin1).opts(color=gold_color, line_width=0.35)

layout3 = (exp_uhv1 * exp_alk1 * exp_alkin1 * err_uhv1 * err_alk1 * err_alkin1).opts(
    title='Experimental: Current',
    xlabel='Distance (Å)',
    ylabel='Current (pA)',
    xlim=(0, 5),
    width=520,
    height=470,
    show_grid=True,
    legend_position='right'
)
layout3

:Overlay
   .Curve.Clean                       :Curve   [z]   (I)
   .Curve.Cont_full_stop_comma_80_pA  :Curve   [z]   (I)
   .Curve.Cont_full_stop_comma_150_pA :Curve   [z]   (I)
   .ErrorBars.I                       :ErrorBars   [x]   (y,yerror)
   .ErrorBars.II                      :ErrorBars   [x]   (y,yerror)
   .ErrorBars.III                     :ErrorBars   [x]   (y,yerror)

In [19]:
# hv.save(layout3, 'fig4_exp_decay.html')

In [20]:
# Experimental plot 2: Decay constant with error bars
exp_uhv2 = exp_uhv_df.hvplot(x='z', y='dI', label='Clean', color=uhv_color, line_width=line_width)
exp_alk2 = exp_alk_df.hvplot(x='z', y='dI', label='Cont., 80 pA', line_width=line_width)
exp_alkin2 = exp_alkin_df.hvplot(x='z', y='dI', label='Cont., 150 pA', color=gold_color, line_width=line_width)

# Create error bars
errors_uhv2 = [(iz_uhv[0][i], iz_uhv[3][i], uhv_err_der[i]) for i in range(len(uhv_err_der))]
errors_alk2 = [(iz_alk[0][i], iz_alk[3][i], alk_err_der[i]) for i in range(len(alk_err_der))]
errors_alkin2 = [(iz_alkin[0][i], iz_alkin[3][i], alkin_err_der[i]) for i in range(len(alkin_err_der))]

err_uhv2 = hv.ErrorBars(errors_uhv2).opts(color=uhv_color, line_width=0.35)
err_alk2 = hv.ErrorBars(errors_alk2).opts(color=red_color, line_width=0.35)
err_alkin2 = hv.ErrorBars(errors_alkin2).opts(color=gold_color, line_width=0.35)

layout4 = (exp_uhv2 * exp_alk2 * exp_alkin2 * err_uhv2 * err_alk2 * err_alkin2).opts(
    title='Experimental: Decay Constant',
    xlabel='Distance (Å)',
    ylabel='Decay constant, κ (1/Å)',
    xlim=(0, 1),
    ylim=(0, 2.8),
    width=520,
    height=470,
    show_grid=True,
    legend_position='right'
)
layout4

:Overlay
   .Curve.Clean                       :Curve   [z]   (dI)
   .Curve.Cont_full_stop_comma_80_pA  :Curve   [z]   (dI)
   .Curve.Cont_full_stop_comma_150_pA :Curve   [z]   (dI)
   .ErrorBars.I                       :ErrorBars   [x]   (y,yerror)
   .ErrorBars.II                      :ErrorBars   [x]   (y,yerror)
   .ErrorBars.III                     :ErrorBars   [x]   (y,yerror)

In [21]:
# hv.save(layout4, 'fig4_exp_kappa.html')